# 05. InfraScan Baseline Crack Classifier (ResNet-50 Transfer Learning)
This notebook trains a transfer-learning binary crack classifier (ResNet-50) using PyTorch on a Google Colab T4 GPU.

**Trained Datasets:**
1. `concrete_crack_cls`
2. `sdnet2018`

**Key Pipeline Specifications:**
- **Shared Preprocessing:** 224x224 letterbox, ImageNet normalization, CLAHE contrast enhancement (`app.ai.preprocessing`).
- **Train Augmentation (pre-preprocess):** Horizontal/vertical flips, 90-degree rotations, +/-20% brightness/contrast jitter (no aggressive crops).
- **Loss Function:** `BCEWithLogitsLoss` with `pos_weight = (no-crack count / crack count)` computed strictly from the `train` split.
- **Phase 1 (2 epochs):** Backbone frozen, linear head only, AdamW (lr=1e-3).
- **Phase 2 (up to 12 epochs):** Full fine-tuning, AdamW (lr=1e-4, weight_decay=1e-4), Cosine Annealing, Early stopping patience=3 on validation F1.
- **Mixed Precision:** PyTorch `autocast` + `GradScaler` on T4 GPU.
- **Validation Threshold:** Maximizes F1 subject to Recall >= 0.90 (evaluated strictly on validation set).
- **Checkpoints & Artifacts:** Saved to Google Drive under `/content/drive/MyDrive/infrascan/models/classification/<dataset>/`.


### 1. Verify GPU Availability
Check CUDA device status and GPU name (recommended: T4).

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[WARNING] CUDA is not available. Please switch runtime to T4 GPU (Runtime -> Change runtime type).")


### 2. Mount Google Drive
Mount Google Drive to persist trained checkpoints, plots, and model cards.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
drive_base = "/content/drive/MyDrive/infrascan/models/classification"
os.makedirs(drive_base, exist_ok=True)
print(f"Google Drive output directory ready: {drive_base}")


### 3. Extract Classification Data & Code
Unpack `classification_data.zip` and `ml_code.zip` to local fast SSD storage `/content`.

In [ ]:
import zipfile
import os
from pathlib import Path

# Paths to the uploaded zip packages (either in /content or in Google Drive)
data_zip_candidates = [
    "/content/classification_data.zip",
    "/content/drive/MyDrive/infrascan/classification_data.zip",
]
code_zip_candidates = [
    "/content/ml_code.zip",
    "/content/drive/MyDrive/infrascan/ml_code.zip",
]

def find_file(candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

data_zip = find_file(data_zip_candidates)
code_zip = find_file(code_zip_candidates)

print(f"Data package found: {data_zip}")
print(f"Code package found: {code_zip}")

# 1. Unzip ML code into /content
if code_zip and os.path.exists(code_zip):
    print("Unzipping ML code to /content...")
    with zipfile.ZipFile(code_zip, "r") as zf:
        zf.extractall("/content")
    print("ML code extracted.")
else:
    print("[Action Required] Upload ml_code.zip to /content or Drive.")

# 2. Unzip dataset images into /content/data/raw
if data_zip and os.path.exists(data_zip):
    print("Unzipping dataset to /content/data/raw...")
    with zipfile.ZipFile(data_zip, "r") as zf:
        zf.extractall("/content/data/raw")
    print("Dataset extracted.")
else:
    print("[Action Required] Upload classification_data.zip to /content or Drive.")


### 4. Setup Python Path & Verify Imports
Ensure `/content` is on `sys.path` and import training modules.

In [ ]:
import sys
import os

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from ml.classification import train_classifier, config
from app.ai.preprocessing import preprocess, PreprocessConfig

print("[OK] Successfully imported train_classifier, config, and preprocess.")
print(f"Model Input Size: {config.MODEL_INPUT_SIZE}")
print(f"Normalize Mode:   {config.NORMALIZE_MODE}")
print(f"Batch Size:       {config.BATCH_SIZE}")


### 5. Run Fast Debug Test (2,000 train / 500 val, 1 epoch per phase)
Verify the end-to-end training pipeline and GPU execution in ~1 minute before full runs.

In [ ]:
data_root = "/content/data/raw"
manifest_path = "/content/data/raw/manifests_colab/concrete_crack_cls.csv"
debug_output_dir = "/content/debug_run"

train_df, val_df, _ = train_classifier.load_manifest(manifest_path, data_root=data_root)

debug_summary = train_classifier.train(
    train_df=train_df,
    val_df=val_df,
    output_dir=debug_output_dir,
    dataset_name="concrete_crack_cls_debug",
    manifest_name="concrete_crack_cls.csv",
    debug=True,
    use_clahe=True,
)
print("Debug test run completed successfully!")


### 6. Full Training: Concrete Crack Classification Dataset
Train ResNet-50 across the complete Concrete Crack Classification dataset. Outputs are saved to Google Drive.

In [ ]:
data_root = "/content/data/raw"
manifest_concrete = "/content/data/raw/manifests_colab/concrete_crack_cls.csv"
output_concrete = "/content/drive/MyDrive/infrascan/models/classification/concrete_crack_cls"

train_df_c, val_df_c, _ = train_classifier.load_manifest(manifest_concrete, data_root=data_root)

summary_concrete = train_classifier.train(
    train_df=train_df_c,
    val_df=val_df_c,
    output_dir=output_concrete,
    dataset_name="concrete_crack_cls",
    manifest_name="concrete_crack_cls.csv",
    debug=False,
    use_clahe=True,
)


### 7. Concrete Crack Classification Summary & Saved Artifacts
Display performance summary, optimal decision threshold, and list saved files.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

output_dir = Path("/content/drive/MyDrive/infrascan/models/classification/concrete_crack_cls")
card_path = output_dir / "model_card.json"

if card_path.exists():
    with open(card_path, "r") as f:
        card = json.load(f)
    print("=" * 60)
    print("CONCRETE CRACK CLASSIFICATION MODEL CARD:")
    print("=" * 60)
    print(f"Best Epoch:        {card['best_epoch']}")
    print(f"Optimal Threshold: {card['chosen_threshold']}")
    print(f"Validation F1:     {card['validation_metrics_at_threshold']['f1_score']}")
    print(f"Validation Recall: {card['validation_metrics_at_threshold']['recall']}")
    print(f"Validation Prec:   {card['validation_metrics_at_threshold']['precision']}")
    print(f"Validation Acc:    {card['validation_metrics_at_threshold']['accuracy']}")
    print(f"Validation AUC:    {card['validation_metrics_at_threshold']['auc_roc']}")
    print("=" * 60)

print("\nSaved Artifacts:")
for item in sorted(output_dir.iterdir()):
    size_mb = item.stat().st_size / (1024 * 1024)
    print(f"  - {item.name:<25} ({size_mb:.2f} MB)")

if (output_dir / "training_curves.png").exists():
    display(Image(str(output_dir / "training_curves.png")))
if (output_dir / "val_confusion_matrix.png").exists():
    display(Image(str(output_dir / "val_confusion_matrix.png")))


### 8. Full Training: SDNET2018 Dataset
Train ResNet-50 across the complete SDNET2018 dataset (handling bridge decks, pavements, and walls). Outputs are saved to Google Drive.

In [ ]:
data_root = "/content/data/raw"
manifest_sdnet = "/content/data/raw/manifests_colab/sdnet2018.csv"
output_sdnet = "/content/drive/MyDrive/infrascan/models/classification/sdnet2018"

train_df_s, val_df_s, _ = train_classifier.load_manifest(manifest_sdnet, data_root=data_root)

summary_sdnet = train_classifier.train(
    train_df=train_df_s,
    val_df=val_df_s,
    output_dir=output_sdnet,
    dataset_name="sdnet2018",
    manifest_name="sdnet2018.csv",
    debug=False,
    use_clahe=True,
)


### 9. SDNET2018 Summary & Saved Artifacts
Display performance summary, optimal decision threshold, and list saved files.

In [ ]:
output_dir = Path("/content/drive/MyDrive/infrascan/models/classification/sdnet2018")
card_path = output_dir / "model_card.json"

if card_path.exists():
    with open(card_path, "r") as f:
        card = json.load(f)
    print("=" * 60)
    print("SDNET2018 CLASSIFICATION MODEL CARD:")
    print("=" * 60)
    print(f"Best Epoch:        {card['best_epoch']}")
    print(f"Optimal Threshold: {card['chosen_threshold']}")
    print(f"Validation F1:     {card['validation_metrics_at_threshold']['f1_score']}")
    print(f"Validation Recall: {card['validation_metrics_at_threshold']['recall']}")
    print(f"Validation Prec:   {card['validation_metrics_at_threshold']['precision']}")
    print(f"Validation Acc:    {card['validation_metrics_at_threshold']['accuracy']}")
    print(f"Validation AUC:    {card['validation_metrics_at_threshold']['auc_roc']}")
    print("=" * 60)

print("\nSaved Artifacts:")
for item in sorted(output_dir.iterdir()):
    size_mb = item.stat().st_size / (1024 * 1024)
    print(f"  - {item.name:<25} ({size_mb:.2f} MB)")

if (output_dir / "training_curves.png").exists():
    display(Image(str(output_dir / "training_curves.png")))
if (output_dir / "val_confusion_matrix.png").exists():
    display(Image(str(output_dir / "val_confusion_matrix.png")))
